# Project 09

In [2]:
#imports
import math
import numpy as np

#TODO: add error checking to check P(x) in forward and in backwards is equal
#TODO: pulling matrix setup and traceback into their own reusable methods
#TODO: handle recursion limit failing on longer sequences - for loop
#TODO:  converting all the probabilities to logs once in __init__ and storing them (np.log), instead of calling math.log inside the loop. 

#TODO: handle chr22 N regions in HMM error checking segment & in the same check run Viterbi with the non-N chunks of sequence, retaining the correct positions.
    # OR adding N to each state’s emissions with the same probability in every state.
#TODO: add f strings to errors
#TODO: feed in another chr data using chr 22 test, or compare decoded path to the UCSC islands as a results measure of how well the model works.
#TODO: README add in rationale for using chr 22

## How would you account for the large memory requirements as the chromosome data grows larger? How would you validate the results your model produces?


Forward algorithm
Initialize with the initial probabilities and the first observation.
Recursively calculate joint probabilities by summing over all possible previous states.
Compute the total sequence probability by summing the final column.

Backward algorithm
Initialize the final position with a value of 1 for all states.
Recursively calculate conditional probabilities, working from the end to the beginning.
Combine with the initial probabilities to get the sequence likelihood.

Forward–Backward algorithm
Calculate both the forward and backward matrices.
Combine them to compute the posterior marginal probability for each state at each position.
Normalize to ensure a valid probability distribution.

In [3]:
def train_cpg_hmm(fasta_path, cpg_path, chrom="chr22", pseudocount=1.0):
    """Estimate probabilities for states CpG island (I) and Genomic (G)."""
    # Uppercase sequences and store in array as bytes
    with open(fasta_path) as fasta_file:
        seq = "".join(line.strip() for line in fasta_file if not line.startswith(">")).upper()
    bases = np.frombuffer(seq.encode(), dtype="S1")

    # Define CpG as True using UCSC CpG file cols 3 and 4
    is_island = np.zeros(len(bases), dtype=bool)
    with open(cpg_path) as cpg_file:
        for line in cpg_file:
            columns = line.split("\t")
            if columns[1] == chrom:
                is_island[int(columns[2]):int(columns[3])] = True

    # Do not accept N letter bases
    valid_base = np.isin(bases, [b"A", b"C", b"G", b"T"])   
    # Create distinction between CpG island and genomic regions
    masks = {"I": valid_base & is_island, "G": valid_base & ~is_island}
    # Define normalization object which takes a dict of counts and returns same keys with values as proportions instead of counts
    normalize = lambda counts: {key: float(value / sum(counts.values())) for key, value in counts.items()}

    # Calculate initial probabilities for each state, overall fraction of each state
    state_counts = {}
    for state, mask in masks.items():
        state_counts[state] = np.count_nonzero(mask)
    init_probs = normalize(state_counts)

    # Calculate emission probabilities, how likely is each base given each state
    # Adding in pseudocount of 1 to prevent any prob from being 0 which would fail in Viterbi log calculation
    emit_probs = {}
    for state, mask in masks.items():
        bases_in_state = bases[mask]
        base_counts = {base: np.count_nonzero(bases_in_state == base.encode()) + pseudocount for base in "ACGT"}
        emit_probs[state] = normalize(base_counts)

    # Transition counts between adjacent bases, how often the sequence moves from one state to the next
    # Line up each position with the one after it, every column is a transition. Ensure bases are non-N
    both_valid = valid_base[:-1] & valid_base[1:]
    from_island = is_island[:-1][both_valid]
    to_island = is_island[1:][both_valid]
    # True is 1 and False is 0, multiply by 2 to keep "to" and "from" states from being mixed up
    transition_codes = 2 * from_island + to_island
    # Count how many times each transition code appears in 4 slot array, rows are the “from” state and columns are the “to” state
    (g_to_g, g_to_i), (i_to_g, i_to_i) = np.bincount(transition_codes, minlength=4).reshape(2, 2) + pseudocount
    # Normalize “from” state’s counts to probs for every row summing to 1
    trans_probs = {"I": normalize({"I": i_to_i, "G": i_to_g}), "G": normalize({"I": g_to_i, "G": g_to_g})}

    return init_probs, trans_probs, emit_probs

In [8]:
class HiddenMarkovModel:
    """Hidden Markov Model for genomic sequence decoding."""

    @staticmethod
    def _checksum(name: str, prob: dict) -> None:
        """
        Check to make sure all probabilities add up to 1. Error checking.
        """
        total = sum(prob.values())
        if not math.isclose(total, 1.0, abs_tol=1e-9):  # general relative tolerance for math.isclose()
            raise ValueError("{name} must sum to 1. Got sum: {total}")


    def __init__(
        self,
        init_probs: dict[str, float],
        trans_probs: dict[str, dict[str, float]],
        emit_probs: dict[str, dict[str, float]],
    ) -> None:
        """
        Initialize the HMM object and set parameters.

        Args:
            init_probs: {state: P(start in state)}
            trans_probs: {from_state: {to_state: P(to_state | from_state)}}
            emit_probs: {state: {symbol: P(symbol | state)}}
        """
        self.init_probs = init_probs
        self.trans_probs = trans_probs
        self.emit_probs = emit_probs
        self.states = list(emit_probs.keys())

        # TODO: dictionaries and use np.logaddexp

        #---------------------------------
        # ERROR CHECK INPUTS
        # empty probs
        if not init_probs or not trans_probs or not emit_probs:
            raise ValueError("init_probs, trans_probs, emit_probs must be non-empty")
        
        # every state needs to be represented in all probs
        if set(emit_probs) != set(trans_probs) or set(emit_probs) != set(init_probs) or set(init_probs) != set(trans_probs):
            raise ValueError("States differ across init_probs, emit_probs, and trans_probs")
        
        # all probabilities add up to 1
        self._checksum("init_probs", init_probs)
        for state, row in trans_probs.items():
            self._checksum(f"trans_probs[{state!r}]", row)
        for state, row in emit_probs.items():
            self._checksum(f"emit_probs[{state!r}]", row)

        # trans_probs references unknown state
        for state, row in trans_probs.items():
            if set(row) != set(self.states):
                raise ValueError("trans_probs[{state}] references unknown state")
        #-----------------------------------


    def _viterbi_recursion(self, seq, prob_matrix, traceback):
        # Next position
        next_pos = len(prob_matrix)

        # Recursion end condition
        if next_pos == len(seq):
            return prob_matrix, traceback

        prev = prob_matrix[next_pos - 1]
        probs = {}
        pointers = {}
        for s in self.states:
            # Calculate maximum probability (best previous state p, maximizing score at t-1 + transition p -> s)
            best_prev = None
            best_score = float("-inf")
            for p in self.states:
                score = prev[p] + math.log(self.trans_probs[p][s])
                if score > best_score:
                    best_prev, best_score = p, score

            # Apply transition and emission probabilities at each step
            probs[s] = (
                prev[best_prev]
                + math.log(self.trans_probs[best_prev][s])
                + math.log(self.emit_probs[s][seq[next_pos]])
            )
            pointers[s] = best_prev

        # Store probability and traceback pointer
        prob_matrix.append(probs)
        traceback.append(pointers)

        return self._viterbi_recursion(seq, prob_matrix, traceback)

    def _viterbi_traceback(self, traceback, last_state):
        # Trace back through the matrix to reconstruct the optimal path
        path = [last_state]
        for t in range(len(traceback) - 1, 0, -1):
            path.append(traceback[t][path[-1]])
        path.reverse()
        return "".join(path)

    def viterbi(self, seq: str) -> tuple[str, float]:        
        # Initialization
        # Set up probability matrix using initial probabilities and the first observation
        # Initialize a traceback matrix for path reconstruction
        first_row = {}
        for s in self.states:
            first_row[s] = math.log(self.init_probs[s]) + math.log(self.emit_probs[s][seq[0]])
        prob_matrix = [first_row]

        traceback = [{s: None for s in self.states}]

        # Recursion
        prob_matrix, traceback = self._viterbi_recursion(seq, prob_matrix, traceback)

        # Identify the final state with the highest probability
        last_row = prob_matrix[-1]
        last_state = None
        for s in self.states:
            if last_state is None or last_row[s] > last_row[last_state]:
                last_state = s

        log_prob = prob_matrix[-1][last_state]

        # Find most likely path
        path = self._viterbi_traceback(traceback, last_state)

        return path, log_prob
    
    def forward()
        # initialization same as viterbi
        # recursion is same as viterbi except sum 
        # termiantion
        

        pass

    def backwards()
        # initialization is 1 
        
        pass

    def forward_backwards()
        # call forward, call backward, also check P(x) is the same in both or raise error
        # two arrays and P(x) inputs, outputs final array


        pass






In [9]:
# Test with Toy Dataset

# Example initial probabilities (probability of starting in each state)
init_probs = {
    "I": 0.2,
    "G": 0.8
}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {
    "I": {"I": 0.7, "G": 0.3},
    "G": {"I": 0.1, "G": 0.9}
}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {
    "I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
    "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}
}

toy_hmm = HiddenMarkovModel(init_probs, trans_probs, emit_probs)
print(f"State path for input 'GGCACTGAA': {toy_hmm.viterbi("GGCACTGAA")}")
print(f"State path for input 'ATATCGCGCGCGAT': {toy_hmm.viterbi("ATATCGCGCGCGAT")}")

State path for input 'GGCACTGAA': ('GGGGGGGGG', -13.929108456051065)
State path for input 'ATATCGCGCGCGAT': ('GGGGIIIIIIIIGG', -21.20203079978548)


In [13]:
## Driver Program with Chr 22 Dataset 

init_probs, trans_probs, emit_probs = train_cpg_hmm("/Users/juliannemurthy/Downloads/chr22.fa", "/Users/juliannemurthy/Downloads/cpgIslandExt.txt")
# Create the model and store probs inside of it
chr22_hmm = HiddenMarkovModel(init_probs, trans_probs, emit_probs)

print("Model built with states:", chr22_hmm.states)

print("\nInitial probabilities")
for state, prob in init_probs.items():
    print(f"  {state}: {prob:.4f}")

print("\nTransition probabilities")
for from_state, row in trans_probs.items():
    for to_state, prob in row.items():
        print(f"  {from_state} -> {to_state}: {prob:.6f}")

print("\nEmission probabilities")
for state, row in emit_probs.items():
    base_probs = "   ".join(f"{base}: {prob:.4f}" for base, prob in row.items())
    print(f"  {state}: {base_probs}   (GC = {row['G'] + row['C']:.1%})")

# Run the chr22 HMM model 
with open("/Users/juliannemurthy/Downloads/chr22.fa") as fasta_file:
    sequence = "".join(line.strip() for line in fasta_file if not line.startswith(">")).upper()
print(f"\nLoaded chr22: {len(sequence):,} bases")

# Run Viterbi on the whole chromosome
path, log_prob = chr22_hmm.viterbi(sequence)

Model built with states: ['I', 'G']

Initial probabilities
  I: 0.0151
  G: 0.9849

Transition probabilities
  I -> I: 0.998751
  I -> G: 0.001249
  G -> I: 0.000019
  G -> G: 0.999981

Emission probabilities
  I: A: 0.1469   C: 0.3533   G: 0.3529   T: 0.1469   (GC = 70.6%)
  G: A: 0.2669   C: 0.2321   G: 0.2343   T: 0.2666   (GC = 46.6%)

Loaded chr22: 50,818,468 bases


KeyError: 'N'